In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike
import numpyro
import numpyro.distributions as dist

jax.config.update("jax_enable_x64", True)

import altair as alt

from mixres.sim import PoissonDisjoint1D
from mixres.models._inference import run_inference_mcmc

from bspline1d_utils import (
    make_piecewise_bspline_basis,
    make_block_difference_matrix,
    get_segment_diff_sizes,
)
from interval_utils import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_interval_sum_matrix,
)

/Users/shozen/Imperial/0_Research/mixres/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Generation

In [3]:
# Generate the dataset
df = PoissonDisjoint1D(cut_points=[18, 64], n_observations=100, seed=42).generate()
df_expanded = expand_interval_dataframe(df, interval_col="interval", x_col="x")

## Setup

In [ ]:
BASIS_DENSITY = 0.8
BOUNDARY_EXT = 0.5
PENALTY_ORDER = 1

In [ ]:
df_model = df.groupby(["interval_id"])["y"].agg(N="size", y="sum").reset_index()

x = np.arange(0, 101)
y_obs = df_model["y"].values
interval_id = df_model["interval_id"].values
N = df_model["N"].values

intervals = df["interval"].unique().tolist()
A = build_interval_sum_matrix(intervals)
interval_width = compute_interval_widths(intervals)

# Piecewise spline basis matching DGP cut points [18, 64] -> [0,17], [18,63], [64,100].
# basis_density=0.5 gives 18 points -> 9 basis functions in the first segment.
B, segment_basis_counts = make_piecewise_bspline_basis(
    x,
    cut_points=[18, 64],
    basis_density=BASIS_DENSITY,
    boundary_ext=BOUNDARY_EXT,
    return_basis_counts=True,
)
segment_diff_sizes = get_segment_diff_sizes(segment_basis_counts, order=PENALTY_ORDER)

# Map each x point to its spline segment index (0, 1, 2)
segment_id_x = np.select([x < 18, x < 64], [0, 1], default=2).astype(np.int32)
n_segments = int(segment_id_x.max() + 1)

# Convert to JAX arrays
y_obs = jnp.array(y_obs)
interval_id = jnp.array(interval_id)
N = jnp.array(N)
A_jax = jnp.array(A)
B_jax = jnp.array(B)
interval_width_jax = jnp.array(interval_width)
segment_id_x_jax = jnp.array(segment_id_x)
D_jax = jnp.array(
    make_block_difference_matrix(segment_basis_counts, order=PENALTY_ORDER)
)

In [6]:
def plot_posterior_lam(df_expanded, mcmc_samples, cut_points):
    lam = mcmc_samples["lam"].mean(axis=0)
    lam_lower = jnp.percentile(mcmc_samples["lam"], 2.5, axis=0)
    lam_upper = jnp.percentile(mcmc_samples["lam"], 97.5, axis=0)

    df_results = df_expanded[["x", "lambda"]].drop_duplicates().reset_index(drop=True)
    df_results["lam"] = np.array(lam)
    df_results["lam_lower"] = np.array(lam_lower)
    df_results["lam_upper"] = np.array(lam_upper)
    df_results["segment"] = np.select(
        [df_results["x"] < cp for cp in cut_points],
        list(range(len(cut_points))),
        default=len(cut_points),
    )

    true_line = (
        alt.Chart(df_results)
        .mark_line(color="red")
        .encode(x="x", y=alt.Y("lambda:Q", title="lambda"), detail="segment:N")
    )
    mean_line = (
        alt.Chart(df_results)
        .mark_line(color="blue")
        .encode(x="x", y=alt.Y("lam:Q", title="lambda"), detail="segment:N")
    )
    bands = (
        alt.Chart(df_results)
        .mark_area(color="lightblue", opacity=0.5)
        .encode(
            x="x",
            y=alt.Y("lam_lower:Q", title="lambda"),
            y2=alt.Y2("lam_upper:Q", title="lambda"),
            detail="segment:N",
        )
    )
    return true_line + mean_line + bands

## Experiment 1: B-Spline Regression

In [7]:
def model_bspline(
    A: ArrayLike,
    B: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    segment_id_x: ArrayLike,
    n_segments: int,
    y_obs=None,
):
    """
    Bayesian B-spline regression model with per-segment intercepts.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        alpha  ~ N(0, 1)^S          per-segment intercepts (S = number of segments)
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, alpha, w ~ Poisson(exp(beta0 + alpha[segment] + B @ w))
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    alpha = numpyro.sample("alpha", dist.Normal(0, 1).expand([n_segments]).to_event(1))
    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + alpha[segment_id_x] + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [8]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_bspline,
    A=A_jax,
    B=B_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_id_x=segment_id_x_jax,
    n_segments=n_segments,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 2082.77it/s, 63 steps of size 7.85e-02. acc. prob=0.93]


Number of divergences: 0
dict_keys(['alpha', 'beta0', 'lam', 'w'])


In [9]:
plot_posterior_lam(df_expanded, mcmc_samples, cut_points=[18, 64])

alt.LayerChart(...)

## Experiment 2: P-Spline Regression

In [10]:
def model_pspline(
    A: ArrayLike,
    B: ArrayLike,
    D: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    segment_diff_sizes: list[int],
    segment_id_x: ArrayLike,
    y_obs: ArrayLike = None,
    epsilon: float = 1e-3,
):
    """
    Bayesian P-spline regression model with segment-wise independent difference priors
    and per-segment intercepts.
    """
    K = B.shape[1]
    n_segments = len(segment_diff_sizes)

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    alpha = numpyro.sample("alpha", dist.Normal(0, 1).expand([n_segments]).to_event(1))

    # Independent difference priors by disjoint spline segment.
    delta_segments = []
    for seg_idx, seg_size in enumerate(segment_diff_sizes):
        delta_seg = numpyro.sample(
            f"delta_seg_{seg_idx}",
            dist.Normal(0, 1).expand([seg_size]).to_event(1),
        )
        delta_segments.append(delta_seg)
    delta = jnp.concatenate(delta_segments)

    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # Soft constraints on finite differences
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + alpha[segment_id_x] + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [11]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_pspline,
    A=A_jax,
    B=B_jax,
    D=D_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_diff_sizes=segment_diff_sizes,
    segment_id_x=segment_id_x_jax,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:07<00:00, 132.49it/s, 1023 steps of size 5.04e-04. acc. prob=0.89]

Number of divergences: 0
dict_keys(['alpha', 'beta0', 'delta_seg_0', 'delta_seg_1', 'delta_seg_2', 'lam', 'w'])


In [12]:
plot_posterior_lam(df_expanded, mcmc_samples, cut_points=[18, 64])

alt.LayerChart(...)

## Experiment 3: Finnish Horseshoe P-Spline

In [13]:
def model_fHS(
    A: ArrayLike,
    B: ArrayLike,
    D: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    segment_diff_sizes: list[int],
    segment_id_x: ArrayLike,
    y_obs: ArrayLike = None,
    epsilon: float = 1e-3,
    slab_scale: float = 2.0,
    slab_df: float = 4.0,
):
    """
    Bayesian P-spline model with segment-wise independent regularized horseshoe priors
    and per-segment intercepts.

    The regularized (Finnish) horseshoe (Piironen & Vehtari, 2017) adds a Student-t
    slab that clips the tails of the local shrinkage scales, preventing the extreme
    values that cause divergences in HMC with the standard horseshoe.

    For each disjoint spline segment:
        tau_seg  ~ HalfCauchy(1)              global scale
        lambda_seg ~ HalfCauchy(1)            local scales
        c2_seg   ~ InvGamma(slab_df/2,
                             slab_df/2 * slab_scale^2)  slab variance
        lambda_tilde_seg = sqrt(
            c2_seg * lambda_seg^2 / (c2_seg + tau_seg^2 * lambda_seg^2)
        )
        delta_seg = z_seg * tau_seg * lambda_tilde_seg,  z_seg ~ N(0, 1)
    """
    K = B.shape[1]
    n_segments = len(segment_diff_sizes)

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    alpha = numpyro.sample("alpha", dist.Normal(0, 1).expand([n_segments]).to_event(1))

    # Independent regularized horseshoe priors by disjoint spline segment.
    delta_segments = []
    for seg_idx, seg_size in enumerate(segment_diff_sizes):
        # Global and local shrinkage scales
        tau_seg = numpyro.sample(f"tau_seg_{seg_idx}", dist.HalfCauchy(1.0))
        lambda_seg = numpyro.sample(
            f"lambda_seg_{seg_idx}",
            dist.HalfCauchy(1.0).expand([seg_size]).to_event(1),
        )
        # Slab variance: regularizes tails of lambda to prevent divergences
        c2_seg = numpyro.sample(
            f"c2_seg_{seg_idx}",
            dist.InverseGamma(slab_df / 2.0, slab_df / 2.0 * slab_scale**2),
        )
        # Regularized local scale (clipped towards slab_scale when lambda is large)
        lambda_tilde_seg = jnp.sqrt(
            c2_seg * lambda_seg**2 / (c2_seg + tau_seg**2 * lambda_seg**2)
        )
        z_seg = numpyro.sample(
            f"delta_raw_seg_{seg_idx}",
            dist.Normal(0, 1).expand([seg_size]).to_event(1),
        )
        delta_seg = z_seg * tau_seg * lambda_tilde_seg
        delta_segments.append(delta_seg)
    delta = jnp.concatenate(delta_segments)

    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # Soft constraints on finite differences
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + alpha[segment_id_x] + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [14]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_fHS,
    A=A_jax,
    B=B_jax,
    D=D_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_diff_sizes=segment_diff_sizes,
    segment_id_x=segment_id_x_jax,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:16<00:00, 62.14it/s, 1023 steps of size 5.83e-04. acc. prob=0.93]

Number of divergences: 9
dict_keys(['alpha', 'beta0', 'c2_seg_0', 'c2_seg_1', 'c2_seg_2', 'delta_raw_seg_0', 'delta_raw_seg_1', 'delta_raw_seg_2', 'lam', 'lambda_seg_0', 'lambda_seg_1', 'lambda_seg_2', 'tau_seg_0', 'tau_seg_1', 'tau_seg_2', 'w'])


In [15]:
plot_posterior_lam(df_expanded, mcmc_samples, cut_points=[18, 64])

alt.LayerChart(...)